# Agentic keyword-assignment workflow

This workflow uses the model as the planner:
1. retrieve relevant examples
2. ask the model for candidate keywords
3. map each candidate to the canonical vocabulary
4. return the final mapped result as a DataFrame

In [ ]:
from typing import Any

import pandas as pd
import weaviate
from pydantic import BaseModel, Field
from pydantic_ai import Agent, RunContext
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider
from pydantic_ai.capabilities import Thinking

from src.simpleRetriever import simpleRetriever
from src.mapping import Mapping

# --- model setup for vLLM OpenAI-compatible API ---
model = OpenAIChatModel(
    "Qwen/Qwen3.5-35B-A3B",
    provider=OpenAIProvider(
        base_url="http://localhost:9513/v1",
        api_key="unused",
    ),
)

class KeywordAssignmentResult(BaseModel):
    # Normierte_Schlagworte: list[str] = Field(..., description="Finale normierte Schlagworte aus dem kanonischen Vokabular")
    # Freie_Schlagworte: list[str] = Field(..., description="Vom Modell vorgeschlagene freie Schlagworte")
    normed_subjects: list[dict[str, Any]] = Field(default_factory=list, description="Normierte Schlagworte zusammen mit ihren IDs aus dem kanonischen Vokabular")

agent = Agent(
    model,
    output_type=KeywordAssignmentResult,
    system_prompt=(
        "Du bist ein Assistent für die Sacherschließung in einer wissenschaftlichen Bibliothek."
        "Verwende retrieve(), um ähnliche Beispieltexte mit Schlagworten abzurufen, und schlage anschließend "
        "Kandidatenschlagwörter für den Eingabetext vor. "
        "Verwende dann map(), um jedes Kandidatenschlagwort auf das kanonische "
        "Schlagwortvokabular abzubilden. "
        "Prüfe, ob die abgebildeten  weiterhin zum Eingabetext passen, "
        "und schließe irrelevante oder falsche Zuordnungen aus. "
        "Gib die finalen abgebildeten Schlagwörter als JSON mit ihren label_id Nummern zurück."
    ),
    capabilities=[Thinking()],
)

@agent.tool
def retrieve(ctx: RunContext[None], text: str, n_examples: int = 5) -> list[dict]:
    retriever = simpleRetriever(
        input_text=text,
        n_examples=n_examples,
        collection_name="title_train",
        tei_port="8090",
        weaviate_port="8087",
    )
    return retriever.retrieve_examples().to_dict(orient="records")

@agent.tool
def map(ctx: RunContext[None], candidate: str) -> dict:
    client = weaviate.connect_to_local(port=8087)
    try:
        mapper = Mapping(
            hyperparameters={
                "host": "8090",
                "alpha": 0.7,
                "use_phrase": False,
                "search": "hybrid",
            },
            collection_name="ki_fsprompt_vocab",
            phrase=None,
            debug=False,
            db_connection=client,
        )
        result = mapper.query_vector_database(candidate=candidate)
        item = next(iter(result.values()))
        return {
            "candidate": candidate,
            **item,
        }
    finally:
        client.close()



In [4]:
from src.simpleRetriever import simpleRetriever

def retrieve_test(text: str, n_examples: int = 5) -> list[dict]:
    retriever = simpleRetriever(
        input_text=text,
        n_examples=n_examples,
        collection_name="title_train",
        tei_port="8090",
        weaviate_port="8087",
    )
    return retriever.retrieve_examples().to_dict(orient="records")

retrieve_test(text="Funktionentheorie - Eine Einführung", n_examples=5)

[{'retrieved_text': 'Die Phiale - zur zeichenhaften Funktion eines Gefäßtyps',
  'retrieved_label_texts': 'Ritual; Vasenmalerei; Akropolis Athen (Athen); Griechenland (Altertum); Omphalosschale; Verwendung; Phiale (Motiv); Phiale',
  'distance': 0.3958323001861572},
 {'retrieved_text': 'Die Phiale - zur zeichenhaften Funktion eines Gefäßtyps',
  'retrieved_label_texts': 'Ritual; Vasenmalerei; Akropolis Athen (Athen); Griechenland (Altertum); Omphalosschale; Verwendung; Phiale (Motiv); Phiale',
  'distance': 0.3958323001861572},
 {'retrieved_text': 'Neuroökonomie : eine wissenschaftstheoretische Analyse',
  'retrieved_label_texts': 'Neuroökonomik',
  'distance': 0.4193035960197449},
 {'retrieved_text': 'Schule nach Parsons. Auf dem Weg zu einer normativ-funktionalistischen Schultheorie',
  'retrieved_label_texts': 'Pädagogische Soziologie; Schultheorie; Parsons, Talcott (1902-1979)',
  'distance': 0.4237368106842041},
 {'retrieved_text': 'Eine realzeitfähige Architektur zur Integrat

In [6]:
text = "Der Platz des Publikums,Kunst und Öffentlichkeit im 18. Jahrhundert"


result = await agent.run(text)

print("MESSAGES")
for m in result.all_messages():
    print(m)

print("OUTPUT")
print(result.output)

MESSAGES
ModelRequest(parts=[SystemPromptPart(content='Du bist ein Assistent für die Sacherschließung in einer wissenschaftlichen Bibliothek.Verwende retrieve(), um ähnliche Beispieltexte mit Schlagworten abzurufen, und schlage anschließend Kandidatenschlagwörter für den Eingabetext vor. Verwende dann map(), um jedes Kandidatenschlagwort auf das kanonische Schlagwortvokabular abzubilden. Prüfe, ob die abgebildeten  weiterhin zum Eingabetext passen, und schließe irrelevante oder falsche Zuordnungen aus. Gib die finalen abgebildeten Schlagwörter als JSON mit ihren label_id Nummern zurück.', timestamp=datetime.datetime(2026, 9, 25, 14, 2, 0, 412572, tzinfo=datetime.timezone.utc)), UserPromptPart(content='Der Platz des Publikums,Kunst und Öffentlichkeit im 18. Jahrhundert', timestamp=datetime.datetime(2026, 9, 25, 14, 2, 0, 412591, tzinfo=datetime.timezone.utc))], timestamp=datetime.datetime(2026, 9, 25, 14, 2, 0, 413814, tzinfo=datetime.timezone.utc), run_id='01a0d8df-7d56-702d-82d1-ff44

In [5]:
from pydantic_ai.messages import ModelRequest, ModelResponse, ToolCallPart, ToolReturnPart

# Walk the message history and print exactly what args the model sent to each tool,
# and what the tool returned, in call order.
for message in result.all_messages():
    if isinstance(message, ModelResponse):
        for part in message.parts:
            if isinstance(part, ToolCallPart):
                print(f"CALL {part.tool_name}({part.args})")
    if isinstance(message, ModelRequest):
        for part in message.parts:
            if isinstance(part, ToolReturnPart):
                print(f"RETURN {part.tool_name} -> {part.content}")

CALL retrieve({"text": "Funktionentheorie - Eine Einführung", "n_examples": 5})
RETURN retrieve -> [{'retrieved_text': 'Die Phiale - zur zeichenhaften Funktion eines Gefäßtyps', 'retrieved_label_texts': 'Ritual; Vasenmalerei; Akropolis Athen (Athen); Griechenland (Altertum); Omphalosschale; Verwendung; Phiale (Motiv); Phiale', 'distance': 0.3958323001861572}, {'retrieved_text': 'Die Phiale - zur zeichenhaften Funktion eines Gefäßtyps', 'retrieved_label_texts': 'Ritual; Vasenmalerei; Akropolis Athen (Athen); Griechenland (Altertum); Omphalosschale; Verwendung; Phiale (Motiv); Phiale', 'distance': 0.3958323001861572}, {'retrieved_text': 'Neuroökonomie : eine wissenschaftstheoretische Analyse', 'retrieved_label_texts': 'Neuroökonomik', 'distance': 0.4193035960197449}, {'retrieved_text': 'Schule nach Parsons. Auf dem Weg zu einer normativ-funktionalistischen Schultheorie', 'retrieved_label_texts': 'Pädagogische Soziologie; Schultheorie; Parsons, Talcott (1902-1979)', 'distance': 0.42373